In [11]:
from __future__ import annotations

import os
import sys
import time
import json
import random
import traceback
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Callable, Optional

import pandas as pd


In [12]:
# ---------------------------------------------------------------------------
# 0. CONFIG
# ---------------------------------------------------------------------------

# Ten-year window per the dissertation methodology.
START_DATE = "2016-01-01"
END_DATE   = "2025-12-31"

# Where all outputs are written.
DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Providers tried in order for each symbol. vnstock routes these through
# different Vietnamese data providers; if one provider is rate-limited or
# temporarily down, the next one is tried. Adjust the order to taste.
VNSTOCK_SOURCES = ["KBS", "VCI", "SSI", "VND", "FPTS", "TCBS"]

# Minimum trading days required to keep a ticker in the analytical sample.
# 1500 days ≈ 6 years, ensuring at least two full walk-forward test windows
# and coverage of multiple regime cycles (2018 correction, COVID-19, 2022 crisis).
# Tickers below this threshold are excluded and logged.
MIN_TRADING_DAYS = 1500

# Retry policy for each individual API call.
MAX_RETRIES     = 3
BACKOFF_BASE    = 2.0  # seconds
RATE_LIMIT_SLEEP = 1.0  # seconds between successful ticker calls

# VN30 constituents as of early 2026. The basket is rebalanced twice a
# year (January and July) so this is a current snapshot; for the full
# historical membership series you should compile HOSE announcements.
# This list is a reasonable starting point and covers the major tickers.
VN30_TICKERS_FALLBACK = [
    "ACB", "BCM", "BID", "BVH", "CTG", "FPT", "GAS", "GVR", "HDB", "HPG",
    "MBB", "MSN", "MWG", "PLX", "POW", "SAB", "SHB", "SSB", "SSI", "STB",
    "TCB", "TPB", "VCB", "VHM", "VIB", "VIC", "VJC", "VNM", "VPB", "VRE",
]

In [13]:
# ---------------------------------------------------------------------------
# 1. UTILITIES
# ---------------------------------------------------------------------------

def log(msg: str, level: str = "INFO") -> None:
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {level:5s} {msg}", flush=True)


def with_retry(fn: Callable, *args, label: str = "", **kwargs):
    """Call fn with exponential backoff. Returns None on permanent failure."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            wait = BACKOFF_BASE ** attempt + random.random()
            log(
                f"{label} attempt {attempt}/{MAX_RETRIES} failed: "
                f"{type(e).__name__}: {e}. Retrying in {wait:.1f}s",
                level="WARN",
            )
            time.sleep(wait)
    log(f"{label} permanently failed after {MAX_RETRIES} attempts", level="ERROR")
    return None


def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    log(f"Saved {len(df):>6d} rows to {path}")


def _ensure_time_column(df: pd.DataFrame) -> pd.DataFrame:
    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce")
        return df

    for c in ("date", "datetime", "trading_date", "tradingdate", "index"):
        if c in df.columns:
            df["time"] = pd.to_datetime(df[c], errors="coerce")
            return df

    if isinstance(df.index, pd.DatetimeIndex):
        df = df.copy()
        df["time"] = pd.to_datetime(df.index, errors="coerce")

    return df


def load_parquet_if_exists(
    path: Path,
    required_cols: Optional[list[str]] = None,
    label: str = "cache",
) -> Optional[pd.DataFrame]:
    if not path.exists():
        return None

    df = pd.read_parquet(path)
    df.columns = [str(c).lower() if isinstance(c, str) else c for c in df.columns]

    if required_cols:
        if "time" in required_cols:
            df = _ensure_time_column(df)
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            log(
                f"Ignoring stale {label} at {path}: missing columns {missing}",
                level="WARN",
            )
            return None

    return df


def _log_frame_diagnostic(
    name: str,
    df: Optional[pd.DataFrame],
    required_cols: Optional[list[str]] = None,
) -> None:
    if df is None:
        log(f"{name}: value is None", level="WARN")
        return

    if df.empty:
        log(f"{name}: dataframe is empty", level="WARN")
        return

    cols = [str(c) for c in df.columns]
    log(f"{name}: rows={len(df)}, cols={len(cols)}, columns={cols}")

    if required_cols:
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            log(f"{name}: missing required columns {missing}", level="WARN")

    if "time" in df.columns:
        t = pd.to_datetime(df["time"], errors="coerce")
        valid = int(t.notna().sum())
        if valid > 0:
            log(
                f"{name}: time coverage {t.min().date()} -> {t.max().date()} "
                f"({valid}/{len(t)} valid)",
            )
        else:
            log(f"{name}: 'time' exists but all values are invalid", level="WARN")
    else:
        log(f"{name}: missing 'time' column", level="WARN")


def stage_5_input_diagnostic(
    ohlcv: Optional[pd.DataFrame],
    vnindex: Optional[pd.DataFrame],
    vix: Optional[pd.DataFrame],
) -> None:
    log("STAGE 5 PRECHECK: input diagnostics")
    _log_frame_diagnostic(
        "VN-Index",
        vnindex,
        required_cols=["time", "open", "high", "low", "close", "volume", "symbol", "source"],
    )
    _log_frame_diagnostic(
        "CBOE VIX",
        vix,
        required_cols=["time", "open", "high", "low", "close", "symbol", "source"],
    )
    _log_frame_diagnostic(
        "VN30 OHLCV",
        ohlcv,
        required_cols=["time", "symbol", "open", "high", "low", "close", "volume", "source"],
    )

In [14]:
# ---------------------------------------------------------------------------
# STAGE 0 - Environment / vnstock smoke test
# ---------------------------------------------------------------------------
# Runs a minimal one-ticker fetch to confirm vnstock is installed, the
# network reaches the provider, and the API shape is what we expect.
# If this stage fails, stop and fix the environment before continuing.

def stage_0_environment_check() -> bool:
    log("STAGE 0: environment check")
    try:
        import vnstock  # noqa: F401
        log(f"vnstock version: {getattr(vnstock, '__version__', 'unknown')}")
    except ImportError:
        log("vnstock is not installed. Run: pip install vnstock", level="ERROR")
        return False

    # Try one tiny call to confirm connectivity.
    try:
        df = fetch_ohlcv_single("VNM", start="2025-01-02", end="2025-01-10")
    except Exception:
        log("Smoke test raised an exception", level="ERROR")
        traceback.print_exc()
        return False

    if df is None or df.empty:
        log("Smoke test returned empty data - provider may be down", level="ERROR")
        return False

    log(f"Smoke test OK, received {len(df)} rows for VNM")
    return True


# ---------------------------------------------------------------------------
# CORE FETCH HELPERS
# ---------------------------------------------------------------------------
# A ticker is fetched by trying each provider in VNSTOCK_SOURCES in order.
# If every provider fails, fall back to yfinance with the ".VN" suffix.

def fetch_ohlcv_single(
    symbol: str,
    start: str = START_DATE,
    end: str = END_DATE,
) -> Optional[pd.DataFrame]:
    """Try every provider for a single symbol and return the first success."""

    # --- Attempt A: vnstock v3 API ---
    try:
        from vnstock import Vnstock
        for source in VNSTOCK_SOURCES:
            def _call():
                stock = Vnstock().stock(symbol=symbol, source=source)
                return stock.quote.history(start=start, end=end)

            df = with_retry(_call, label=f"{symbol} via vnstock/{source}")
            if df is not None and not df.empty:
                df = _standardise(df, symbol, source)
                return df
    except ImportError:
        pass

    # --- Attempt B: vnstock v2 legacy API ---
    try:
        from vnstock import stock_historical_data
        def _call_v2():
            return stock_historical_data(
                symbol=symbol, start_date=start, end_date=end
            )
        df = with_retry(_call_v2, label=f"{symbol} via vnstock v2 legacy")
        if df is not None and not df.empty:
            return _standardise(df, symbol, "vnstock_v2")
    except ImportError:
        pass

    # --- Attempt C: yfinance fallback ---
    try:
        import yfinance as yf
        def _call_yf():
            return yf.download(
                f"{symbol}.VN", start=start, end=end, progress=False
            )
        df = with_retry(_call_yf, label=f"{symbol} via yfinance")
        if df is not None and not df.empty:
            df = df.reset_index()
            df.columns = [c.lower() if isinstance(c, str) else c for c in df.columns]
            df = df.rename(columns={"date": "time"})
            return _standardise(df, symbol, "yfinance")
    except ImportError:
        pass

    return None


def _standardise(df: pd.DataFrame, symbol: str, source: str) -> pd.DataFrame:
    """Return a canonical schema: time, symbol, open, high, low, close, volume, source."""
    df = df.copy()
    df.columns = [str(c).lower() for c in df.columns]

    # Common time column names across providers
    for c in ("time", "date", "trading_date", "tradingdate"):
        if c in df.columns:
            df["time"] = pd.to_datetime(df[c])
            break

    keep = ["time", "open", "high", "low", "close", "volume"]
    missing = [c for c in keep if c not in df.columns]
    if missing:
        raise ValueError(f"{symbol}: missing columns {missing} from {source}")

    df = df[keep]
    df["symbol"] = symbol
    df["source"] = source
    return df[["time", "symbol", "open", "high", "low", "close", "volume", "source"]]


In [15]:
# ---------------------------------------------------------------------------
# STAGE 1 - VN30 ticker list
# ---------------------------------------------------------------------------

def stage_1_ticker_list() -> list[str]:
    out_path = DATA_DIR / "vn30_tickers.json"
    log("STAGE 1: resolving VN30 ticker list")

    if out_path.exists():
        tickers = json.loads(out_path.read_text())
        log(f"Loaded {len(tickers)} tickers from cache")
        return tickers

    tickers: list[str] = []
    try:
        # vnstock v3 exposes listing helpers; this may change between versions.
        from vnstock import Vnstock
        listing = Vnstock().stock(symbol="VNINDEX", source="VCI").listing
        df = listing.symbols_by_group("VN30")
        if hasattr(df, "tolist"):
            tickers = df.tolist()
        elif isinstance(df, pd.DataFrame) and "symbol" in df.columns:
            tickers = df["symbol"].tolist()
        elif isinstance(df, pd.Series):
            tickers = df.tolist()
    except Exception as e:
        log(f"Could not fetch VN30 list from vnstock: {e}", level="WARN")

    if not tickers:
        log("Falling back to hardcoded VN30 snapshot", level="WARN")
        tickers = VN30_TICKERS_FALLBACK

    out_path.write_text(json.dumps(sorted(set(tickers)), indent=2))
    log(f"Saved {len(tickers)} tickers to {out_path}")
    return sorted(set(tickers))

In [16]:
# ---------------------------------------------------------------------------
# STAGE 2 - VN-Index benchmark
# ---------------------------------------------------------------------------

def stage_2_vnindex() -> Optional[pd.DataFrame]:
    out_path = DATA_DIR / "vnindex.parquet"
    log("STAGE 2: fetching VN-Index benchmark")

    cached = load_parquet_if_exists(
        out_path,
        required_cols=["time", "open", "high", "low", "close", "volume", "symbol", "source"],
        label="VN-Index cache",
    )
    if cached is not None:
        log(f"Loaded VN-Index from cache ({len(cached)} rows)")
        return cached

    df = fetch_ohlcv_single("VNINDEX", start=START_DATE, end=END_DATE)
    if df is None:
        log("Could not fetch VN-Index - check providers", level="ERROR")
        return None
    save_parquet(df, out_path)
    return df

In [17]:
# ---------------------------------------------------------------------------
# STAGE 3 - VN30 constituent OHLCV (per-ticker checkpointing)
# ---------------------------------------------------------------------------

def stage_3_vn30_ohlcv(tickers: list[str]) -> pd.DataFrame:
    log(f"STAGE 3: fetching OHLCV for {len(tickers)} VN30 constituents")
    per_ticker_dir = DATA_DIR / "tickers"
    per_ticker_dir.mkdir(parents=True, exist_ok=True)

    frames: list[pd.DataFrame] = []
    failed: list[str] = []

    for i, symbol in enumerate(tickers, start=1):
        ticker_path = per_ticker_dir / f"{symbol}.parquet"

        cached = load_parquet_if_exists(
            ticker_path,
            required_cols=["time", "open", "high", "low", "close", "volume", "symbol", "source"],
            label=f"{symbol} cache",
        )
        if cached is not None:
            log(f"[{i:>2}/{len(tickers)}] {symbol}: cached ({len(cached)} rows)")
            frames.append(cached)
            continue

        log(f"[{i:>2}/{len(tickers)}] {symbol}: fetching")
        try:
            df = fetch_ohlcv_single(symbol)
        except Exception:
            traceback.print_exc()
            df = None

        if df is None or df.empty:
            log(f"{symbol}: no data from any provider", level="WARN")
            failed.append(symbol)
            continue

        save_parquet(df, ticker_path)
        frames.append(df)
        time.sleep(RATE_LIMIT_SLEEP)

    if failed:
        log(f"Failed tickers ({len(failed)}): {failed}", level="WARN")

    if not frames:
        log("No ticker data collected - aborting", level="ERROR")
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    combined = combined.sort_values(["symbol", "time"]).reset_index(drop=True)
    save_parquet(combined, DATA_DIR / "vn30_ohlcv.parquet")
    return combined

def stage_3b_coverage_filter(ohlcv: pd.DataFrame) -> pd.DataFrame:
    out_path    = DATA_DIR / "vn30_ohlcv.parquet"
    excl_path   = DATA_DIR / "excluded_tickers.json"
    kept_path   = DATA_DIR / "included_tickers.json"

    log(f"STAGE 3b: applying minimum coverage filter ({MIN_TRADING_DAYS} trading days)")

    if ohlcv is None or ohlcv.empty:
        log("No OHLCV data to filter", level="ERROR")
        return pd.DataFrame()

    # Count trading days per ticker.
    counts = ohlcv.groupby("symbol")["time"].count()

    excluded = counts[counts < MIN_TRADING_DAYS]
    included = counts[counts >= MIN_TRADING_DAYS]

    log(f"Tickers passing filter  : {len(included):>2d}  (>= {MIN_TRADING_DAYS} days)")
    log(f"Tickers excluded        : {len(excluded):>2d}  (<  {MIN_TRADING_DAYS} days)")

    if len(excluded):
        for sym, n in excluded.sort_values().items():
            log(f"  EXCLUDED  {sym:<6s}  {n:>5d} days", level="WARN")

    # Build and save exclusion / inclusion lists with metadata.
    excl_records = [
        {"symbol": sym, "trading_days": int(n), "reason": f"< {MIN_TRADING_DAYS} days"}
        for sym, n in excluded.items()
    ]
    incl_records = [
        {"symbol": sym, "trading_days": int(n)}
        for sym, n in included.sort_values(ascending=False).items()
    ]
    excl_path.write_text(json.dumps(excl_records, indent=2))
    kept_path.write_text(json.dumps(incl_records, indent=2))
    log(f"Exclusion list saved to {excl_path}")
    log(f"Inclusion list saved to {kept_path}")

    # Filter and save.
    filtered = ohlcv[ohlcv["symbol"].isin(included.index)].copy()
    filtered = filtered.sort_values(["symbol", "time"]).reset_index(drop=True)
    save_parquet(filtered, out_path)
    log(f"Filtered dataset: {len(filtered):>7,d} rows | "
        f"{filtered['symbol'].nunique()} tickers -> {out_path}")
    return filtered

In [18]:
# ---------------------------------------------------------------------------
# STAGE 4 - Global risk proxy (CBOE VIX via yfinance)
# ---------------------------------------------------------------------------

def stage_4_vix() -> Optional[pd.DataFrame]:
    out_path = DATA_DIR / "vix.parquet"
    log("STAGE 4: fetching CBOE VIX")

    cached = load_parquet_if_exists(
        out_path,
        required_cols=["time", "symbol", "source", "open", "high", "low", "close"],
        label="VIX cache",
    )
    if cached is not None:
        log(f"Loaded VIX from cache ({len(cached)} rows)")
        return cached

    try:
        import yfinance as yf
    except ImportError:
        log("yfinance not installed. Run: pip install yfinance", level="ERROR")
        return None

    def _call():
        return yf.download("^VIX", start=START_DATE, end=END_DATE, progress=False)

    df = with_retry(_call, label="VIX via yfinance")
    if df is None or df.empty:
        log("Could not fetch VIX", level="ERROR")
        return None

    df = df.reset_index()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [str(c[0]).lower() for c in df.columns]
    else:
        df.columns = [str(c).lower() for c in df.columns]

    df = _ensure_time_column(df)
    if "time" not in df.columns:
        log("VIX payload missing time/date column", level="ERROR")
        return None

    df = df[df["time"].notna()].copy()
    df["symbol"] = "VIX"
    df["source"] = "yfinance"
    cols = ["time", "symbol", "open", "high", "low", "close", "volume", "source"]
    df = df[[c for c in cols if c in df.columns]]
    save_parquet(df, out_path)
    return df

In [19]:
# ---------------------------------------------------------------------------
# STAGE 5 - Data quality report
# ---------------------------------------------------------------------------

def stage_5_quality_report(
    ohlcv_raw: pd.DataFrame,
    ohlcv_filtered: pd.DataFrame,
    vnindex: Optional[pd.DataFrame],
    vix: Optional[pd.DataFrame],
) -> None:
    log("STAGE 5: data quality report")
    out_path = DATA_DIR / "quality_report.txt"
    lines = []

    def add(s: str) -> None:
        lines.append(s)
        print(s)

    add("=" * 60)
    add(f"Date range requested : {START_DATE} -> {END_DATE}")
    add(f"Min trading days     : {MIN_TRADING_DAYS}")
    add("=" * 60)

    # --- Benchmarks ---
    if vnindex is not None and not vnindex.empty:
        add(f"VN-Index   : {len(vnindex):>6d} rows  "
            f"[{vnindex['time'].min().date()} -> {vnindex['time'].max().date()}]")
    else:
        add("VN-Index   : MISSING")

    if vix is not None and not vix.empty:
        add(f"CBOE VIX   : {len(vix):>6d} rows  "
            f"[{vix['time'].min().date()} -> {vix['time'].max().date()}]")
    else:
        add("CBOE VIX   : MISSING")

    # --- Raw (pre-filter) ---
    if ohlcv_raw is not None and not ohlcv_raw.empty:
        add(f"\nRAW (before filter)  : {len(ohlcv_raw):>6d} rows | "
            f"{ohlcv_raw['symbol'].nunique()} tickers")
        raw_counts = (
            ohlcv_raw.groupby("symbol")["time"]
            .agg(["count", "min", "max"])
            .rename(columns={"count": "days", "min": "first", "max": "last"})
            .sort_values("days", ascending=False)
        )
        add("\nPer-ticker raw counts:")
        for sym, row in raw_counts.iterrows():
            marker = "  EXCLUDED" if row["days"] < MIN_TRADING_DAYS else ""
            add(f"  {sym:<6s} {row['days']:>5d} days  "
                f"[{row['first'].date()} -> {row['last'].date()}]{marker}")
    else:
        add("VN30 OHLCV RAW : MISSING")

    # --- Filtered (post-filter) ---
    if ohlcv_filtered is not None and not ohlcv_filtered.empty:
        add(f"\nFILTERED (analytical) : {len(ohlcv_filtered):>6d} rows | "
            f"{ohlcv_filtered['symbol'].nunique()} tickers")
    else:
        add("VN30 OHLCV FILTERED : MISSING")

    # --- Excluded tickers detail ---
    excl_path = DATA_DIR / "excluded_tickers.json"
    if excl_path.exists():
        excluded = json.loads(excl_path.read_text())
        if excluded:
            add(f"\nExcluded tickers ({len(excluded)}):")
            for rec in excluded:
                add(f"  {rec['symbol']:<6s} {rec['trading_days']:>5d} days — {rec['reason']}")
        else:
            add("\nNo tickers excluded.")

    out_path.write_text("\n".join(lines))
    log(f"Quality report written to {out_path}")

In [20]:
# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

def main() -> int:
    log("=" * 60)
    log("VN30 data pull - starter pipeline")
    log("=" * 60)

    if not stage_0_environment_check():
        log("Environment check failed - aborting", level="ERROR")
        return 1

    tickers = stage_1_ticker_list()
    vnindex = stage_2_vnindex()
    ohlcv_raw    = stage_3_vn30_ohlcv(tickers)
    ohlcv        = stage_3b_coverage_filter(ohlcv_raw)
    vix     = stage_4_vix()
    stage_5_input_diagnostic(ohlcv, vnindex, vix)
    stage_5_quality_report(ohlcv_raw, ohlcv, vnindex, vix)

    log("=" * 60)
    log("Pipeline complete")
    log(f"Analytical dataset : {DATA_DIR / 'vn30_ohlcv.parquet'}")
    log(f"Excluded tickers   : {DATA_DIR / 'excluded_tickers.json'}")
    log(f"Quality report     : {DATA_DIR / 'quality_report.txt'}")
    log("=" * 60)
    return 0


if __name__ == "__main__":
    # In notebooks, avoid SystemExit so the cell reports success.
    if "get_ipython" in globals():
        main()
    else:
        sys.exit(main())


[07:08:00] INFO  ============================================================
[07:08:00] INFO  VN30 data pull - starter pipeline
[07:08:00] INFO  ============================================================
[07:08:00] INFO  STAGE 0: environment check
[07:08:00] INFO  vnstock version: unknown
[07:08:01] INFO  Smoke test OK, received 7 rows for VNM
[07:08:01] INFO  STAGE 1: resolving VN30 ticker list
[07:08:01] INFO  Loaded 30 tickers from cache
[07:08:01] INFO  STAGE 2: fetching VN-Index benchmark
[07:08:01] INFO  Loaded VN-Index from cache (2611 rows)
[07:08:01] INFO  STAGE 3: fetching OHLCV for 30 VN30 constituents
[07:08:01] INFO  [ 1/30] ACB: cached (2611 rows)
[07:08:01] INFO  [ 2/30] BID: cached (2611 rows)
[07:08:01] INFO  [ 3/30] CTG: cached (2611 rows)
[07:08:01] INFO  [ 4/30] DGC: cached (2611 rows)
[07:08:01] INFO  [ 5/30] FPT: cached (2611 rows)
[07:08:01] INFO  [ 6/30] GAS: cached (2611 rows)
[07:08:01] INFO  [ 7/30] GVR: cached (1942 rows)
[07:08:01] INFO  [ 8/30] HDB: cac